# Modelling - SEMMA

In [1]:
import sqlite3
import pandas as pd
from pathlib import Path
import numpy as np
from scipy.stats import ks_2samp

## SAMPLE (Amostragem)

### Extraindo  os Dados do Banco

In [2]:
DB_PRINCIPAL = "../../data/processed/feature_store.db"   
DB_DIVIDAS   = "../../data/processed/database.db"         

query = Path("target.sql").read_text(encoding="utf-8")

con = sqlite3.connect(DB_PRINCIPAL)

try:
    con.execute(f"ATTACH DATABASE '{DB_DIVIDAS}' AS dividas")
    df = pd.read_sql_query(query, con)
finally:
    con.close()

print(df.shape)
print(df["target_ever90_mob12"].mean())

(104623, 299)
0.0347820268965715


In [3]:
df.to_parquet('../../data/processed/abt.parquet')

### Split de Treinamento

In [4]:
df = pd.read_parquet('../../data/processed/abt.parquet')

In [5]:
df['data_ref'] = pd.to_datetime(df['data_ref'])

data_corte = '2023-06-01'

df_train = df[df['data_ref'] < data_corte].copy()
df_oot = df[df['data_ref'] >= data_corte].copy()

print(f"Dimensões Treino: {df_train.shape}")
print(f"Dimensões OOT: {df_oot.shape}")

print(f"Interval de Tempo Treino: {df_train['data_ref'].min()} até {df_train['data_ref'].max()}")
print(f"Interval de Tempo OOT: {df_oot['data_ref'].min()} até {df_oot['data_ref'].max()}")

print(f"Inadimplência Treino: {df_train['target_ever90_mob12'].mean() * 100:.2f}%")
print(f"Inadimplência OOT: {df_oot['target_ever90_mob12'].mean() * 100:.2f}%")


Dimensões Treino: (82836, 299)
Dimensões OOT: (21787, 299)
Interval de Tempo Treino: 2017-02-04 00:00:00 até 2023-05-31 00:00:00
Interval de Tempo OOT: 2023-06-01 00:00:00 até 2024-03-05 00:00:00
Inadimplência Treino: 3.22%
Inadimplência OOT: 4.45%


In [6]:
X_train = df_train.drop(columns=['target_ever90_mob12'])
y_train = df_train['target_ever90_mob12']

X_oot = df_oot.drop(columns=['target_ever90_mob12'])
y_oot = df_oot['target_ever90_mob12']

## Explore (Exploração)

### Variáveis Númericas

In [7]:
correlations = X_train.select_dtypes(include=['number']).corrwith(y_train)
correlations.sort_values(ascending=False).head(20)


c:\Users\patry\AppData\Local\miniconda3\envs\case_datarisk\lib\site-packages\numpy\lib\_function_base_impl.py:3045: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
c:\Users\patry\AppData\Local\miniconda3\envs\case_datarisk\lib\site-packages\numpy\lib\_function_base_impl.py:3046: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


flag_cliente_sem_historico          0.049172
flag_credito_acima_max_historico    0.029530
pct_recusados_30d                   0.027646
flag_primeira_vez_tipo_produto      0.026071
pct_pagamentos_parciais             0.021993
pct_atraso_5d_3m                    0.020226
pct_recusados_120d                  0.018311
dias_desde_ultimo_pagamento         0.017548
pct_recusados_90d                   0.017458
pct_recusa_desistencia_cliente      0.016628
flag_novo_tipo_contrato             0.016611
flag_primeira_vez_tipo_contrato     0.016611
variacao_parcela_vs_media_6m        0.016146
pct_atraso_5d_6m                    0.016030
dias_desde_ultimo_contrato          0.015325
pct_parcelas_atrasadas_6m           0.015290
dias_desde_ultimo_aprovado          0.014991
pct_recusados_6m                    0.014703
desvio_credito_vs_ultimo            0.013841
pct_recusados_12m                   0.013547
dtype: float64

In [8]:
correlations = X_train.select_dtypes(include=['number']).corrwith(y_train)
correlations.sort_values(ascending=True).head(20)

c:\Users\patry\AppData\Local\miniconda3\envs\case_datarisk\lib\site-packages\numpy\lib\_function_base_impl.py:3045: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
c:\Users\patry\AppData\Local\miniconda3\envs\case_datarisk\lib\site-packages\numpy\lib\_function_base_impl.py:3046: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


pct_aprovados_30d                -0.052660
qtd_contratos_anteriores         -0.047438
qtd_aprovados                    -0.047438
qtd_parcelas_vencidas_vida       -0.044894
qtd_total                        -0.044514
qtd_areas_distintas              -0.043500
qtd_parcelas_vencidas_12m        -0.043426
qtd_parcelas_vencidas_6m         -0.041278
razao_pago_previsto              -0.039679
qtd_parcelas_vencidas_3m         -0.039532
prazo_restante_medio_aberto      -0.038437
qtd_contratos_ativos_estimado    -0.038151
qtd_contratos_abertos_parc       -0.037886
qtd_produto_nao_informado        -0.037798
qtd_tipos_produto_distintos      -0.036613
qtd_tipo_emprestimo_dinheiro     -0.035482
qtd_tipo_financiamento_consumo   -0.035033
qtd_aprovados_12m                -0.033513
valor_credito_soma               -0.033019
qtd_tipo_credito_rotativo        -0.032028
dtype: float64

In [9]:


ks_stats = {
    col: ks_2samp(X_train.loc[y_train == 1, col], X_train.loc[y_train == 0, col]).statistic
    for col in X_train.select_dtypes(include=['number']).columns
}
pd.Series(ks_stats).sort_values(ascending=False).head(20)


valor_credito_soma                0.140326
qtd_contratos_anteriores          0.139594
qtd_aprovados                     0.139594
qtd_parcelas_vencidas_vida        0.139423
qtd_total                         0.137626
flag_cliente_sem_historico        0.137626
qtd_areas_distintas               0.132483
qtd_produto_nao_informado         0.125563
qtd_parcelas_vencidas_12m         0.124587
qtd_parcelas_vencidas_6m          0.117472
qtd_contratos_ativos_estimado     0.113454
saldo_aberto_estimado             0.112240
qtd_contratos_abertos_parc        0.111652
parcela_mensal_aberta_parc        0.111328
qtd_parcelas_vencidas_3m          0.110271
qtd_tipo_financiamento_consumo    0.107634
qtd_tipo_emprestimo_dinheiro      0.101446
qtd_total_12m                     0.101221
qtd_aprovados_12m                 0.095186
valor_credito_soma_12m            0.091785
dtype: float64

In [10]:

df_train_explore = X_train.copy()
df_train_explore["target"] = y_train

describe = (
    df_train_explore
    .groupby("target")
    .mean(numeric_only=True)
    .T
)

describe["variable"] = describe.index
describe["ratio"] = describe[1] / describe[0].replace(0, np.nan)

cols = ["variable"] + [col for col in describe.columns if col != "variable"]
describe = describe[cols]


pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)
pd.set_option('display.max_rows', None)
display(describe)
     

target,variable,0,1,ratio
id_contrato,id_contrato,2.041774e+06,2.050372e+06,1.004211
id_cliente,id_cliente,3.044436e+05,3.074231e+05,1.009787
valor_credito,valor_credito,1.626154e+05,1.496856e+05,0.920489
valor_bem,valor_bem,1.616717e+05,1.478455e+05,0.914480
valor_parcela,valor_parcela,1.354059e+04,1.337403e+04,0.987700
flag_fim_de_semana,flag_fim_de_semana,0.000000e+00,0.000000e+00,NaN
hora,hora,1.258772e+01,1.261671e+01,1.002303
flag_sem_bem,flag_sem_bem,7.594147e-02,7.980517e-02,1.050877
valor_bem_sobre_renda_anual,valor_bem_sobre_renda_anual,1.055503e+00,9.928634e-01,0.940654
entrada_implicita_pct,entrada_implicita_pct,9.345851e-04,1.135749e-03,1.215244


A inadimplência é melhor separada pelo volume de histórico do cliente: inadimplentes têm menos contratos, menos parcelas vencidas e menor saldo/valor de crédito acumulado (KS 0,11 a 0,14), refletindo uma maior proporção de clientes sem histórico (56% vs 42%). 

Em segundo plano, aparecem maior proporção de recusas recentes, crédito acima do máximo histórico e primeira contratação do produto (correlações de 0,02 a 0,03). 

Variáveis de atraso recente em proporção e de desvio em relação ao padrão têm sinal positivo, mas fraco. 

Cadastrais (renda, idade, patrimônio) e identificadores não discriminam. Nenhuma variável isolada passa de KS 0,14 ou |corr| 0,053, então o poder preditivo dependerá da combinação e das interações entre elas.

### Variáveis Categóricas

#### tipo_contrato

In [11]:

print("Frequência relativa de tipo_contrato para target = 0:")
display(
    df_train_explore[df_train_explore['target'] == 0]['tipo_contrato']
    .value_counts(normalize=True)
)

print("\nFrequência relativa de tipo_contrato para target = 1:")
display(
    df_train_explore[df_train_explore['target'] == 1]['tipo_contrato']
    .value_counts(normalize=True)
)


woe_df = (
    df_train_explore
    .groupby('tipo_contrato')['target']
    .value_counts()
    .unstack(fill_value=0)
)

woe_df = woe_df.reindex(columns=[0, 1], fill_value=0)
woe_df.columns = ['good', 'bad']

total_good = woe_df['good'].sum()
total_bad = woe_df['bad'].sum()

woe_df['dist_good'] = woe_df['good'] / total_good
woe_df['dist_bad'] = woe_df['bad'] / total_bad

epsilon = 1e-10

woe_df['woe'] = np.log(
    (woe_df['dist_good'] + epsilon) /
    (woe_df['dist_bad'] + epsilon)
)

woe_df['iv'] = (
    woe_df['dist_good'] - woe_df['dist_bad']
) * woe_df['woe']

print("\nTabela de WoE e IV por categoria:")
display(woe_df)

iv_total = woe_df['iv'].sum()

print(f"\nInformation Value (IV) total de 'tipo_contrato': {iv_total:.4f}")

Frequência relativa de tipo_contrato para target = 0:


tipo_contrato
Empréstimo em Dinheiro      0.377799
Financiamento de Consumo    0.368855
Crédito Rotativo            0.253346
Name: proportion, dtype: float64


Frequência relativa de tipo_contrato para target = 1:


tipo_contrato
Empréstimo em Dinheiro      0.389284
Financiamento de Consumo    0.377670
Crédito Rotativo            0.233046
Name: proportion, dtype: float64


Tabela de WoE e IV por categoria:


,good,bad,dist_good,dist_bad,woe,iv
tipo_contrato,,,,,,
Crédito Rotativo,20310,622,0.253346,0.233046,0.083520,0.001695
Empréstimo em Dinheiro,30287,1039,0.377799,0.389284,-0.029948,0.000344
Financiamento de Consumo,29570,1008,0.368855,0.377670,-0.023616,0.000208



Information Value (IV) total de 'tipo_contrato': 0.0022


#### faixa_horaria

In [12]:

print("Frequência relativa de faixa_horaria para target = 0:")
display(
    df_train_explore[df_train_explore['target'] == 0]['faixa_horaria']
    .value_counts(normalize=True)
)

print("\nFrequência relativa de faixa_horaria para target = 1:")
display(
    df_train_explore[df_train_explore['target'] == 1]['faixa_horaria']
    .value_counts(normalize=True)
)


woe_df = (
    df_train_explore
    .groupby('faixa_horaria')['target']
    .value_counts()
    .unstack(fill_value=0)
)

woe_df = woe_df.reindex(columns=[0, 1], fill_value=0)
woe_df.columns = ['good', 'bad']

total_good = woe_df['good'].sum()
total_bad = woe_df['bad'].sum()

woe_df['dist_good'] = woe_df['good'] / total_good
woe_df['dist_bad'] = woe_df['bad'] / total_bad

epsilon = 1e-10

woe_df['woe'] = np.log(
    (woe_df['dist_good'] + epsilon) /
    (woe_df['dist_bad'] + epsilon)
)

woe_df['iv'] = (
    woe_df['dist_good'] - woe_df['dist_bad']
) * woe_df['woe']

print("\nTabela de WoE e IV por categoria:")
display(woe_df)

iv_total = woe_df['iv'].sum()

print(f"\nInformation Value (IV) total de 'faixa_horaria': {iv_total:.4f}")

Frequência relativa de faixa_horaria para target = 0:


faixa_horaria
tarde        0.532301
manha        0.378909
noite        0.074470
madrugada    0.014320
Name: proportion, dtype: float64


Frequência relativa de faixa_horaria para target = 1:


faixa_horaria
tarde        0.544773
manha        0.364181
noite        0.074934
madrugada    0.016111
Name: proportion, dtype: float64


Tabela de WoE e IV por categoria:


,good,bad,dist_good,dist_bad,woe,iv
faixa_horaria,,,,,,
madrugada,1148,43,0.014320,0.016111,-0.117832,0.000211
manha,30376,972,0.378909,0.364181,0.039644,0.000584
noite,5970,200,0.074470,0.074934,-0.006223,0.000003
tarde,42673,1454,0.532301,0.544773,-0.023160,0.000289



Information Value (IV) total de 'faixa_horaria': 0.0011


#### estado_civil

In [13]:

print("Frequência relativa de estado_civil para target = 0:")
display(
    df_train_explore[df_train_explore['target'] == 0]['estado_civil']
    .value_counts(normalize=True)
)

print("\nFrequência relativa de estado_civil para target = 1:")
display(
    df_train_explore[df_train_explore['target'] == 1]['estado_civil']
    .value_counts(normalize=True)
)


woe_df = (
    df_train_explore
    .groupby('estado_civil')['target']
    .value_counts()
    .unstack(fill_value=0)
)

woe_df = woe_df.reindex(columns=[0, 1], fill_value=0)
woe_df.columns = ['good', 'bad']

total_good = woe_df['good'].sum()
total_bad = woe_df['bad'].sum()

woe_df['dist_good'] = woe_df['good'] / total_good
woe_df['dist_bad'] = woe_df['bad'] / total_bad

epsilon = 1e-10

woe_df['woe'] = np.log(
    (woe_df['dist_good'] + epsilon) /
    (woe_df['dist_bad'] + epsilon)
)

woe_df['iv'] = (
    woe_df['dist_good'] - woe_df['dist_bad']
) * woe_df['woe']

print("\nTabela de WoE e IV por categoria:")
display(woe_df)

iv_total = woe_df['iv'].sum()

print(f"\nInformation Value (IV) total de 'estado_civil': {iv_total:.4f}")

Frequência relativa de estado_civil para target = 0:


estado_civil
Casado           0.652189
Solteiro         0.129442
União estável    0.095750
Separado         0.066411
Viúvo            0.056208
Name: proportion, dtype: float64


Frequência relativa de estado_civil para target = 1:


estado_civil
Casado           0.636943
Solteiro         0.141251
União estável    0.097415
Separado         0.067066
Viúvo            0.057325
Name: proportion, dtype: float64


Tabela de WoE e IV por categoria:


,good,bad,dist_good,dist_bad,woe,iv
estado_civil,,,,,,
Casado,52284,1700,0.652189,0.636943,0.023654,0.000361
Separado,5324,179,0.066411,0.067066,-0.009814,0.000006
Solteiro,10377,377,0.129442,0.141251,-0.087306,0.001031
União estável,7676,260,0.095750,0.097415,-0.017236,0.000029
Viúvo,4506,153,0.056208,0.057325,-0.019681,0.000022



Information Value (IV) total de 'estado_civil': 0.0014


#### tipo_renda

In [14]:

print("Frequência relativa de tipo_renda para target = 0:")
display(
    df_train_explore[df_train_explore['target'] == 0]['tipo_renda']
    .value_counts(normalize=True)
)

print("\nFrequência relativa de tipo_renda para target = 1:")
display(
    df_train_explore[df_train_explore['target'] == 1]['tipo_renda']
    .value_counts(normalize=True)
)


woe_df = (
    df_train_explore
    .groupby('tipo_renda')['target']
    .value_counts()
    .unstack(fill_value=0)
)

woe_df = woe_df.reindex(columns=[0, 1], fill_value=0)
woe_df.columns = ['good', 'bad']

total_good = woe_df['good'].sum()
total_bad = woe_df['bad'].sum()

woe_df['dist_good'] = woe_df['good'] / total_good
woe_df['dist_bad'] = woe_df['bad'] / total_bad

epsilon = 1e-10

woe_df['woe'] = np.log(
    (woe_df['dist_good'] + epsilon) /
    (woe_df['dist_bad'] + epsilon)
)

woe_df['iv'] = (
    woe_df['dist_good'] - woe_df['dist_bad']
) * woe_df['woe']

print("\nTabela de WoE e IV por categoria:")
display(woe_df)

iv_total = woe_df['iv'].sum()

print(f"\nInformation Value (IV) total de 'tipo_renda': {iv_total:.4f}")

Frequência relativa de tipo_renda para target = 0:


tipo_renda
Assalariado            0.496988
Associado comercial    0.227438
Aposentado             0.195554
Servidor público       0.072025
Autônomo               0.006212
Não informado          0.001647
Desempregado           0.000075
Licença maternidade    0.000050
Estudante              0.000012
Name: proportion, dtype: float64


Frequência relativa de tipo_renda para target = 1:


tipo_renda
Assalariado            0.531285
Associado comercial    0.206819
Aposentado             0.180217
Servidor público       0.075309
Autônomo               0.005620
Não informado          0.000749
Name: proportion, dtype: float64


Tabela de WoE e IV por categoria:


,good,bad,dist_good,dist_bad,woe,iv
tipo_renda,,,,,,
Aposentado,15677,481,0.195554,0.180217,0.081675,0.001253
Assalariado,39842,1418,0.496988,0.531285,-0.066734,0.002289
Associado comercial,18233,552,0.227438,0.206819,0.095032,0.001959
Autônomo,498,15,0.006212,0.005620,0.100142,0.000059
Desempregado,6,0,0.000075,0.000000,13.525744,0.001012
Estudante,1,0,0.000012,0.000000,11.733992,0.000146
Licença maternidade,4,0,0.000050,0.000000,13.120280,0.000655
Não informado,132,2,0.001647,0.000749,0.787247,0.000706
Servidor público,5774,201,0.072025,0.075309,-0.044593,0.000146



Information Value (IV) total de 'tipo_renda': 0.0082


#### nivel_educacao

In [15]:

print("Frequência relativa de nivel_educacao para target = 0:")
display(
    df_train_explore[df_train_explore['target'] == 0]['nivel_educacao']
    .value_counts(normalize=True)
)

print("\nFrequência relativa de nivel_educacao para target = 1:")
display(
    df_train_explore[df_train_explore['target'] == 1]['nivel_educacao']
    .value_counts(normalize=True)
)


woe_df = (
    df_train_explore
    .groupby('nivel_educacao')['target']
    .value_counts()
    .unstack(fill_value=0)
)

woe_df = woe_df.reindex(columns=[0, 1], fill_value=0)
woe_df.columns = ['good', 'bad']

total_good = woe_df['good'].sum()
total_bad = woe_df['bad'].sum()

woe_df['dist_good'] = woe_df['good'] / total_good
woe_df['dist_bad'] = woe_df['bad'] / total_bad

epsilon = 1e-10

woe_df['woe'] = np.log(
    (woe_df['dist_good'] + epsilon) /
    (woe_df['dist_bad'] + epsilon)
)

woe_df['iv'] = (
    woe_df['dist_good'] - woe_df['dist_bad']
) * woe_df['woe']

print("\nTabela de WoE e IV por categoria:")
display(woe_df)

iv_total = woe_df['iv'].sum()

print(f"\nInformation Value (IV) total de 'nivel_educacao': {iv_total:.4f}")

Frequência relativa de nivel_educacao para target = 0:


nivel_educacao
Ensino médio                  0.736537
Ensino superior               0.216486
Ensino superior incompleto    0.032719
Ensino fundamental            0.013060
Pós-graduação                 0.001198
Name: proportion, dtype: float64


Frequência relativa de nivel_educacao para target = 1:


nivel_educacao
Ensino médio                  0.724991
Ensino superior               0.230049
Ensino superior incompleto    0.032222
Ensino fundamental            0.010865
Pós-graduação                 0.001873
Name: proportion, dtype: float64


Tabela de WoE e IV por categoria:


,good,bad,dist_good,dist_bad,woe,iv
nivel_educacao,,,,,,
Ensino fundamental,1047,29,0.013060,0.010865,0.183980,0.000404
Ensino médio,59046,1935,0.736537,0.724991,0.015801,0.000182
Ensino superior,17355,614,0.216486,0.230049,-0.060767,0.000824
Ensino superior incompleto,2623,86,0.032719,0.032222,0.015319,0.000008
Pós-graduação,96,5,0.001198,0.001873,-0.447498,0.000302



Information Value (IV) total de 'nivel_educacao': 0.0017


#### ocupacao

In [16]:

print("Frequência relativa de ocupacao para target = 0:")
display(
    df_train_explore[df_train_explore['target'] == 0]['ocupacao']
    .value_counts(normalize=True)
)

print("\nFrequência relativa de ocupacao para target = 1:")
display(
    df_train_explore[df_train_explore['target'] == 1]['ocupacao']
    .value_counts(normalize=True)
)


woe_df = (
    df_train_explore
    .groupby('ocupacao')['target']
    .value_counts()
    .unstack(fill_value=0)
)

woe_df = woe_df.reindex(columns=[0, 1], fill_value=0)
woe_df.columns = ['good', 'bad']

total_good = woe_df['good'].sum()
total_bad = woe_df['bad'].sum()

woe_df['dist_good'] = woe_df['good'] / total_good
woe_df['dist_bad'] = woe_df['bad'] / total_bad

epsilon = 1e-10

woe_df['woe'] = np.log(
    (woe_df['dist_good'] + epsilon) /
    (woe_df['dist_bad'] + epsilon)
)

woe_df['iv'] = (
    woe_df['dist_good'] - woe_df['dist_bad']
) * woe_df['woe']

print("\nTabela de WoE e IV por categoria:")
display(woe_df)

iv_total = woe_df['iv'].sum()

print(f"\nInformation Value (IV) total de 'ocupacao': {iv_total:.4f}")

Frequência relativa de ocupacao para target = 0:


ocupacao
Operários                          0.180087
Aposentado                         0.177517
Não informado                      0.146157
Equipe de vendas                   0.105729
Equipe administrativa              0.085559
Gerentes                           0.068407
Motoristas                         0.060000
Técnicos especializados            0.038532
Contadores                         0.032557
Profissionais de saúde             0.030299
Equipe de segurança                0.019883
Cozinheiros                        0.017414
Equipe de limpeza                  0.013659
Serviços particulares              0.007285
Operários de baixa qualificação    0.004865
Secretárias                        0.003468
Garçons e atendentes de bar        0.003318
Corretores imobiliários            0.002208
Equipe de tecnologia               0.001509
Equipe de recursos humanos         0.001472
Sem Ocupação                       0.000075
Name: proportion, dtype: float64


Frequência relativa de ocupacao para target = 1:


ocupacao
Operários                          0.189959
Aposentado                         0.161109
Não informado                      0.154365
Equipe de vendas                   0.107156
Equipe administrativa              0.092919
Gerentes                           0.064444
Motoristas                         0.057700
Técnicos especializados            0.032596
Profissionais de saúde             0.031472
Contadores                         0.026227
Equipe de segurança                0.022855
Cozinheiros                        0.016111
Equipe de limpeza                  0.013488
Serviços particulares              0.009741
Garçons e atendentes de bar        0.005245
Operários de baixa qualificação    0.005245
Secretárias                        0.003747
Corretores imobiliários            0.002623
Equipe de recursos humanos         0.001873
Equipe de tecnologia               0.001124
Name: proportion, dtype: float64


Tabela de WoE e IV por categoria:


,good,bad,dist_good,dist_bad,woe,iv
ocupacao,,,,,,
Aposentado,14231,430,0.177517,0.161109,0.096985,0.001591
Contadores,2610,70,0.032557,0.026227,0.216202,0.001369
Corretores imobiliários,177,7,0.002208,0.002623,-0.172168,0.000071
Cozinheiros,1396,43,0.017414,0.016111,0.077758,0.000101
Equipe administrativa,6859,248,0.085559,0.092919,-0.082520,0.000607
Equipe de limpeza,1095,36,0.013659,0.013488,0.012583,0.000002
Equipe de recursos humanos,118,5,0.001472,0.001873,-0.241161,0.000097
Equipe de segurança,1594,61,0.019883,0.022855,-0.139280,0.000414
Equipe de tecnologia,121,3,0.001509,0.001124,0.294770,0.000114



Information Value (IV) total de 'ocupacao': 0.0094


#### tipo_organizacao

In [17]:

print("Frequência relativa de tipo_organizacao para target = 0:")
display(
    df_train_explore[df_train_explore['target'] == 0]['tipo_organizacao']
    .value_counts(normalize=True)
)

print("\nFrequência relativa de tipo_organizacao para target = 1:")
display(
    df_train_explore[df_train_explore['target'] == 1]['tipo_organizacao']
    .value_counts(normalize=True)
)


woe_df = (
    df_train_explore
    .groupby('tipo_organizacao')['target']
    .value_counts()
    .unstack(fill_value=0)
)

woe_df = woe_df.reindex(columns=[0, 1], fill_value=0)
woe_df.columns = ['good', 'bad']

total_good = woe_df['good'].sum()
total_bad = woe_df['bad'].sum()

woe_df['dist_good'] = woe_df['good'] / total_good
woe_df['dist_bad'] = woe_df['bad'] / total_bad

epsilon = 1e-10

woe_df['woe'] = np.log(
    (woe_df['dist_good'] + epsilon) /
    (woe_df['dist_bad'] + epsilon)
)

woe_df['iv'] = (
    woe_df['dist_good'] - woe_df['dist_bad']
) * woe_df['woe']

print("\nTabela de WoE e IV por categoria:")
display(woe_df)

iv_total = woe_df['iv'].sum()

print(f"\nInformation Value (IV) total de 'tipo_organizacao': {iv_total:.4f}")

Frequência relativa de tipo_organizacao para target = 0:


tipo_organizacao
Entidade empresarial tipo 3    0.219317
Não informado                  0.207654
Autônomo                       0.124378
Outro                          0.057630
Governo                        0.041750
Saúde                          0.041488
Escola                         0.034890
Entidade empresarial tipo 2    0.029987
Comércio tipo 7                0.029226
Educação infantil              0.019422
Construção                     0.018187
Entidade empresarial tipo 1    0.016553
Transporte tipo 4              0.014782
Segurança                      0.009568
Indústria tipo 3               0.008557
Indústria tipo 9               0.008358
Habitação                      0.008071
Comércio tipo 3                0.007933
Indústria tipo 11              0.007884
Correios                       0.007023
Polícia                        0.006773
Transporte tipo 2              0.006699
Forças armadas                 0.006636
Agricultura                    0.005364
Banco                  


Frequência relativa de tipo_organizacao para target = 1:


tipo_organizacao
Entidade empresarial tipo 3    0.204196
Não informado                  0.191083
Autônomo                       0.134882
Outro                          0.059198
Governo                        0.044961
Saúde                          0.042713
Escola                         0.040090
Comércio tipo 7                0.028475
Entidade empresarial tipo 2    0.024354
Educação infantil              0.019858
Construção                     0.019108
Entidade empresarial tipo 1    0.015362
Transporte tipo 4              0.014987
Segurança                      0.010865
Forças armadas                 0.010491
Indústria tipo 11              0.010491
Habitação                      0.008617
Indústria tipo 9               0.008243
Transporte tipo 2              0.007868
Restaurante                    0.007493
Correios                       0.007119
Banco                          0.006744
Comércio tipo 3                0.006744
Polícia                        0.006744
Indústria tipo 3       


Tabela de WoE e IV por categoria:


,good,bad,dist_good,dist_bad,woe,iv
tipo_organizacao,,,,,,
Agricultura,430,17,0.005364,0.006369,-0.171836,1.728026e-04
Autônomo,9971,360,0.124378,0.134882,-0.081076,8.516314e-04
Banco,419,18,0.005227,0.006744,-0.254909,3.868267e-04
Celular,49,1,0.000611,0.000375,0.489412,1.157714e-04
Comércio tipo 1,77,2,0.000960,0.000749,0.248250,5.241817e-05
Comércio tipo 2,343,6,0.004279,0.002248,0.643563,1.306777e-03
Comércio tipo 3,636,18,0.007933,0.006744,0.162419,1.931711e-04
Comércio tipo 4,12,0,0.000150,0.000000,14.218891,2.128391e-03
Comércio tipo 5,17,0,0.000212,0.000000,14.567198,3.089081e-03



Information Value (IV) total de 'tipo_organizacao': 0.0381


#### tipo_moradia

In [18]:

print("Frequência relativa de tipo_moradia para target = 0:")
display(
    df_train_explore[df_train_explore['target'] == 0]['tipo_moradia']
    .value_counts(normalize=True)
)

print("\nFrequência relativa de tipo_moradia para target = 1:")
display(
    df_train_explore[df_train_explore['target'] == 1]['tipo_moradia']
    .value_counts(normalize=True)
)


woe_df = (
    df_train_explore
    .groupby('tipo_moradia')['target']
    .value_counts()
    .unstack(fill_value=0)
)

woe_df = woe_df.reindex(columns=[0, 1], fill_value=0)
woe_df.columns = ['good', 'bad']

total_good = woe_df['good'].sum()
total_bad = woe_df['bad'].sum()

woe_df['dist_good'] = woe_df['good'] / total_good
woe_df['dist_bad'] = woe_df['bad'] / total_bad

epsilon = 1e-10

woe_df['woe'] = np.log(
    (woe_df['dist_good'] + epsilon) /
    (woe_df['dist_bad'] + epsilon)
)

woe_df['iv'] = (
    woe_df['dist_good'] - woe_df['dist_bad']
) * woe_df['woe']

print("\nTabela de WoE e IV por categoria:")
display(woe_df)

iv_total = woe_df['iv'].sum()

print(f"\nInformation Value (IV) total de 'tipo_moradia': {iv_total:.4f}")

Frequência relativa de tipo_moradia para target = 0:


tipo_moradia
Casa ou apartamento        0.894171
Mora com os pais           0.042424
Apartamento municipal      0.036037
Imóvel alugado             0.015917
Apartamento funcional      0.008557
Apartamento cooperativo    0.002894
Name: proportion, dtype: float64


Frequência relativa de tipo_moradia para target = 1:


tipo_moradia
Casa ou apartamento        0.896590
Mora com os pais           0.044961
Apartamento municipal      0.031847
Imóvel alugado             0.015362
Apartamento funcional      0.009367
Apartamento cooperativo    0.001873
Name: proportion, dtype: float64


Tabela de WoE e IV por categoria:


,good,bad,dist_good,dist_bad,woe,iv
tipo_moradia,,,,,,
Apartamento cooperativo,232,5,0.002894,0.001873,0.434891,0.000444
Apartamento funcional,686,25,0.008557,0.009367,-0.090406,0.000073
Apartamento municipal,2889,85,0.036037,0.031847,0.123606,0.000518
Casa ou apartamento,71683,2393,0.894171,0.896590,-0.002702,0.000007
Imóvel alugado,1276,41,0.015917,0.015362,0.035505,0.000020
Mora com os pais,3401,120,0.042424,0.044961,-0.058075,0.000147



Information Value (IV) total de 'tipo_moradia': 0.0012


As variáveis categóricas apresentaram IV entre 0,001 e 0,009, todas abaixo do limiar de relevância (0,02). A exceção aparente é tipo_organizacao (IV 0,038), mas esse valor é inflado por categorias com poucos casos e nenhum inadimplente (ex.: Indústria tipo 10 e 13, Comércio tipo 4 e 5), que geram WoE extremo; descontadas, o IV cai para cerca de 0,02. 

Entre categorias com volume suficiente, Forças armadas, Restaurante e Autônomo apresentam risco ligeiramente maior, e Entidade empresarial tipos 2 e 3 e “Não informado” risco ligeiramente menor. 

As variáveis faixa_horaria, tipo_moradia, estado_civil, nivel_educacao e tipo_contrato não discriminam isoladamente. Recomenda-se agrupar categorias raras em “Outros” antes da codificação e priorizar as variáveis de histórico do cliente, de maior poder discriminatório.